In [ ]:
"""
╔══════════════════════════════════════════════════════════════════╗
║   PREMIER LEAGUE 2020-21 — EVENT DATA ANALYSIS PIPELINE         ║
║   Traccia 3: Interpretazione → Feature Engineering → EDA → ML   ║
╚══════════════════════════════════════════════════════════════════╝

Dataset:  https://www.kaggle.com/datasets/shushrutsharma/premier-league-event-data-202021
Struttura: ~607k righe, 35 colonne, evento per evento, 380 partite

Dipendenze:
    pip install pandas pyarrow numpy matplotlib seaborn scikit-learn missingno
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ── sklearn ──────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
from sklearn.pipeline import Pipeline

# ── CONFIG ───────────────────────────────────────────────────────
FILE_PATH = r"C:\Users\Casa\Desktop\Mining_Premier_League\data\dataset.parquet"
OUT_DIR    = Path("pl_plots")
OUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", palette="tab10")

def savefig(name):
    plt.savefig(OUT_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()

def section(title):
    print(f"\n{'═'*60}")
    print(f"  {title}")
    print(f"{'═'*60}")


# ════════════════════════════════════════════════════════════════
# SEZIONE 0 — CARICAMENTO
# ════════════════════════════════════════════════════════════════
section("0. CARICAMENTO E OVERVIEW")

df = pd.read_parquet(FILE_PATH)
print(f"Shape: {df.shape[0]:,} righe × {df.shape[1]} colonne")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nColonne e tipi:")
print(df.dtypes.to_string())


# ════════════════════════════════════════════════════════════════
# SEZIONE 1 — INTERPRETAZIONE DEL DATASET
# ════════════════════════════════════════════════════════════════
section("1. INTERPRETAZIONE DEL DATASET (senza documentazione)")

"""
STRUTTURA INFERITA:
─────────────────────────────────────────────────────────────────
• Ogni riga = 1 evento calcistico (pass, tiro, tackle, fallo…)
• 'Unnamed: 0'    → indice evento INTRA-partita (0-based, si resetta a 0
                     a ogni nuova partita) → usato per costruire match_id
• 'id'            → identificatore univoco dell'evento (float)
• 'eventId'       → ID evento alternativo (int, codifica interna)
• 'minute'/'second' → timestamp dell'evento nel match (32767 = sentinella)
• 'teamId'        → ID numerico della squadra
• 'x', 'y'        → posizione sul campo [0-100] dell'evento
• 'endX','endY'   → posizione di destinazione (solo per pass, tiri, ecc.)
• 'period_value'  → 1=Primo Tempo, 2=Secondo Tempo; period_displayName
• 'type_value/displayName' → TIPO di evento (Pass, Shot, Tackle, …)
• 'outcomeType_value/displayName' → Esito (Successful / Unsuccessful)
• 'isTouch'       → True se l'evento è un contatto con la palla
• 'isShot'        → True SOLO per eventi di tipo tiro
• 'isGoal'        → True SOLO per i gol
• 'isOwnGoal'     → True SOLO per autoreti
• 'goalMouthY/Z'  → posizione porta dove è finito il tiro (solo su tiri)
• 'blockedX/Y'    → coord del blocco (solo su tiri bloccati)
• 'cardType_value/displayName' → tipo cartellino (solo su Card events)
• 'relatedEventId/PlayerId' → collegamento a evento/giocatore correlato
• 'playerId','name','position','shirtNo' → dati giocatore coinvolto
• 'qualifiers'    → lista JSON di qualificatori aggiuntivi (foot, zone…)
• 'satisfiedEventsTypes' → lista di tipi di evento soddisfatti

NULL STRUTTURALI (non da imputare):
  goalMouthZ/Y → null se non è un tiro
  isGoal/isShot/isOwnGoal → null se non è una situazione da gol/tiro
  cardType → null se non è un evento Card
  blockedX/Y → null se il tiro non è bloccato
  endX/Y → null se l'evento non ha destinazione (tackle, fallo, ecc.)
  playerId/name/position → null per eventi di sistema (Start, End, Formation)

SOURCE: Opta/WhoScored format — Premier League 2020-21 (380 partite)
"""

print(__doc__[__doc__.find("STRUTTURA"):__doc__.find("SOURCE")+60])

# Statistiche base per la descrizione
print("\n--- Tipi di evento presenti ---")
print(df["type_displayName"].value_counts().to_string())

print(f"\n--- Periodi ---")
print(df["period_displayName"].value_counts())

print(f"\n--- Valori sentinella in 'minute' ---")
sentinel = df[df["minute"] == 32767]
print(f"  Righe con minute=32767: {len(sentinel):,} → {sentinel['period_displayName'].value_counts().to_dict()}")


# ════════════════════════════════════════════════════════════════
# SEZIONE 2 — COSTRUZIONE MATCH_ID E RISULTATI
# ════════════════════════════════════════════════════════════════
section("2. COSTRUZIONE MATCH_ID E RISULTATI")

# 'Unnamed: 0' si resetta a 0 a ogni nuova partita → 380 reset = 380 match
df = df.sort_index()                          # ordine originale del dataset
df["match_id"] = (df["Unnamed: 0"] == 0).cumsum()

n_matches = df["match_id"].nunique()
print(f"Partite identificate: {n_matches}  (attese: 380 PL 2020-21)")

# Squadre per match (2 per partita)
teams_per_match = df.groupby("match_id")["teamId"].unique()

# Pulizia: rimuovi eventi di sistema (minute=32767) per i calcoli
df_game = df[
    df["period_displayName"].isin(["FirstHalf", "SecondHalf"])
].copy()

# ── Conteggio gol per squadra per match ─────────────────────────
goals_df = (df_game[df_game["type_displayName"] == "Goal"]
            .groupby(["match_id", "teamId"])
            .size()
            .reset_index(name="goals_scored"))

# Matrice match × team con tutte le coppie
all_pairs = (df_game.groupby(["match_id", "teamId"])
             .size()
             .reset_index(name="total_events")
             [["match_id", "teamId"]])

results_df = all_pairs.merge(goals_df, on=["match_id", "teamId"], how="left")
results_df["goals_scored"] = results_df["goals_scored"].fillna(0).astype(int)

# Gol subiti = gol dell'avversario nello stesso match
total_goals = results_df.groupby("match_id")["goals_scored"].transform("sum")
results_df["goals_conceded"] = total_goals - results_df["goals_scored"]
opp = results_df.copy()

def outcome(row):
    if row["goals_scored"] > row["goals_conceded"]:  return "Win"
    elif row["goals_scored"] < row["goals_conceded"]: return "Loss"
    else:                                              return "Draw"

opp["outcome"] = opp.apply(outcome, axis=1)

print("\nDistribuzione outcome (per team-match):")
print(opp["outcome"].value_counts())
print(f"\nGol totali stagione: {opp['goals_scored'].sum() // 2}")  # ogni gol conta 2x

# Check: media gol per partita
avg_goals = opp.groupby("match_id")["goals_scored"].sum().mean()
print(f"Media gol per partita: {avg_goals:.2f}  (PL 2020-21 reale: ~2.7)")


# ════════════════════════════════════════════════════════════════
# SEZIONE 3 — FEATURE EXTRACTION
# ════════════════════════════════════════════════════════════════
section("3. FEATURE EXTRACTION (livello match × team)")

"""
FEATURE ESTRATTE:
─────────────────────────────────────────────────────────────────
Conteggi eventi (per catturare volume e stile di gioco):
  n_passes, n_shots, n_shots_on_target, n_goals,
  n_tackles, n_fouls, n_corners, n_aerials,
  n_clearances, n_interceptions, n_takeons, n_cards,
  n_ball_recoveries

Ratei / percentuali:
  pass_success_rate     = pass riusciti / pass totali
  shot_conversion_rate  = gol / tiri totali
  tackle_success_rate   = tackle riusciti / tackle totali
  takeon_success_rate   = takeon riusciti / takeon totali
  aerial_success_rate   = duelli aerei vinti / duelli totali

Posizionali (proxy di stile tattico):
  avg_x   = posizione media x degli eventi (alto → squadra avanzata)
  avg_y   = posizione media y degli eventi (centro/lato)
  avg_shot_x = posizione media x dei tiri (vicino alla porta = bassa x?)
  possession_proxy = n_passes / (n_passes + n_touches_avversario) *  100

Temporali:
  events_per_minute = densità eventi (intensità ritmo)
  first_half_ratio  = % eventi nel 1° tempo (game management)
"""

# Colonne per calcoli di successo
df_game["is_successful"] = df_game["outcomeType_displayName"] == "Successful"

def extract_features(grp):
    total_events = len(grp)
    minutes_played = grp["minute"].clip(upper=120).max() - grp["minute"].clip(upper=120).min()
    minutes_played = max(minutes_played, 1)

    # Sotto-gruppi per tipo evento
    passes      = grp[grp["type_displayName"] == "Pass"]
    shots       = grp[grp["isShot"].notna()]
    goals       = grp[grp["type_displayName"] == "Goal"]
    tackles     = grp[grp["type_displayName"] == "Tackle"]
    fouls       = grp[grp["type_displayName"] == "Foul"]
    corners     = grp[grp["type_displayName"] == "CornerAwarded"]
    aerials     = grp[grp["type_displayName"] == "Aerial"]
    clearances  = grp[grp["type_displayName"] == "Clearance"]
    intercepts  = grp[grp["type_displayName"] == "Interception"]
    takeons     = grp[grp["type_displayName"] == "TakeOn"]
    cards       = grp[grp["type_displayName"] == "Card"]
    recoveries  = grp[grp["type_displayName"] == "BallRecovery"]
    saved_shots = grp[grp["type_displayName"] == "SavedShot"]
    missed      = grp[grp["type_displayName"] == "MissedShots"]
    on_post     = grp[grp["type_displayName"] == "ShotOnPost"]

    shots_on_target = len(saved_shots) + len(goals)  # parate + gol = in porta

    def safe_rate(num, den):
        return num / den if den > 0 else np.nan

    ft = grp[grp["period_displayName"] == "FirstHalf"]
    st = grp[grp["period_displayName"] == "SecondHalf"]

    return pd.Series({
        # Conteggi
        "n_passes":           len(passes),
        "n_shots":            len(shots),
        "n_shots_on_target":  shots_on_target,
        "n_goals":            len(goals),
        "n_tackles":          len(tackles),
        "n_fouls":            len(fouls),
        "n_corners":          len(corners),
        "n_aerials":          len(aerials),
        "n_clearances":       len(clearances),
        "n_interceptions":    len(intercepts),
        "n_takeons":          len(takeons),
        "n_cards":            len(cards),
        "n_ball_recoveries":  len(recoveries),

        # Ratei
        "pass_success_rate":    safe_rate(passes["is_successful"].sum(), len(passes)),
        "shot_conversion_rate": safe_rate(len(goals), len(shots)),
        "shots_on_target_rate": safe_rate(shots_on_target, len(shots)),
        "tackle_success_rate":  safe_rate(tackles["is_successful"].sum(), len(tackles)),
        "takeon_success_rate":  safe_rate(takeons["is_successful"].sum(), len(takeons)),
        "aerial_success_rate":  safe_rate(aerials["is_successful"].sum(), len(aerials)),

        # Posizionali
        "avg_x":        grp["x"].mean(),
        "avg_y":        grp["y"].mean(),
        "avg_shot_x":   shots["x"].mean() if len(shots) > 0 else np.nan,
        "avg_shot_y":   shots["y"].mean() if len(shots) > 0 else np.nan,

        # Intensità / ritmo
        "events_per_minute": total_events / minutes_played,
        "first_half_ratio":  safe_rate(len(ft), total_events),
    })

print("Estrazione feature in corso...")
features_raw = (df_game.groupby(["match_id", "teamId"])
                .apply(extract_features)
                .reset_index())

# Merge con risultati
features = features_raw.merge(opp[["match_id", "teamId", "goals_scored",
                                    "goals_conceded", "outcome"]],
                               on=["match_id", "teamId"], how="inner")

print(f"Dataset feature finale: {features.shape[0]} righe × {features.shape[1]} colonne")
print(f"Distribuzione outcome finale:")
print(features["outcome"].value_counts())

FEATURE_COLS = [c for c in features.columns
                if c not in ["match_id", "teamId", "goals_scored",
                              "goals_conceded", "outcome"]]
print(f"\nFeature estratte ({len(FEATURE_COLS)}):")
print(FEATURE_COLS)


# ════════════════════════════════════════════════════════════════
# SEZIONE 4 — EDA: CONFRONTO WIN / DRAW / LOSS
# ════════════════════════════════════════════════════════════════
section("4. EDA — CONFRONTO FEATURE PER OUTCOME")

# 4a. Statistiche descrittive per classe
print("\n--- Media per classe (Win / Draw / Loss) ---")
class_means = features.groupby("outcome")[FEATURE_COLS].mean()
print(class_means.round(3).T.to_string())

# 4b. Deviazione standard globale e differenza tra classi
global_std = features[FEATURE_COLS].std()

print("\n--- Feature con differenza WIN-LOSS > 1σ (significative) ---")
diff_win_loss = (class_means.loc["Win"] - class_means.loc["Loss"]).abs()
significant = diff_win_loss[diff_win_loss > global_std]
for feat in significant.index:
    w  = class_means.loc["Win",  feat]
    d  = class_means.loc["Draw", feat] if "Draw" in class_means.index else np.nan
    l  = class_means.loc["Loss", feat]
    sd = global_std[feat]
    print(f"  {feat:<30}  Win={w:.3f}  Draw={d:.3f}  Loss={l:.3f}  |diff|={diff_win_loss[feat]:.3f}  σ={sd:.3f}  → {diff_win_loss[feat]/sd:.1f}σ")

# 4c. Plot: boxplot delle feature più discriminanti
top_features = diff_win_loss.sort_values(ascending=False).head(9).index.tolist()
OUTCOME_ORDER = ["Win", "Draw", "Loss"]
PALETTE = {"Win": "#10B981", "Draw": "#F59E0B", "Loss": "#EF4444"}

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, feat in zip(axes.flatten(), top_features):
    sns.boxplot(data=features, x="outcome", y=feat, order=OUTCOME_ORDER,
                palette=PALETTE, ax=ax, width=0.5, fliersize=3)
    ax.set_title(feat, fontsize=10, fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("")
plt.suptitle("Top 9 feature più discriminanti tra Win / Draw / Loss", fontsize=14, y=1.01)
plt.tight_layout()
savefig("01_boxplot_feature")

# 4d. Heatmap medie normalizzate
means_norm = (class_means - class_means.mean()) / class_means.std()
fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(means_norm.T, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, ax=ax, linewidths=0.5)
ax.set_title("Feature medie normalizzate per classe (z-score)\n↑ Verde = alta per quella classe", fontsize=12)
plt.tight_layout()
savefig("02_heatmap_medie")

# 4e. Violinplot posizionali
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feat in zip(axes, ["avg_x", "avg_y"]):
    sns.violinplot(data=features, x="outcome", y=feat, order=OUTCOME_ORDER,
                   palette=PALETTE, ax=ax, inner="quartile")
    ax.set_title(f"{feat} per outcome")
plt.tight_layout()
savefig("03_violin_posizionale")

# 4f. Scatter: n_shots vs pass_success_rate, colorato per outcome
fig, ax = plt.subplots(figsize=(10, 7))
for outcome_val, grp in features.groupby("outcome"):
    ax.scatter(grp["n_shots"], grp["pass_success_rate"],
               label=outcome_val, alpha=0.5, s=30, color=PALETTE[outcome_val])
ax.set_xlabel("n_shots"); ax.set_ylabel("pass_success_rate")
ax.set_title("Tiri vs Successo nei passaggi, per outcome")
ax.legend()
plt.tight_layout()
savefig("04_scatter_shots_passes")

# 4g. Correlation matrix delle feature
fig, ax = plt.subplots(figsize=(16, 13))
corr = features[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.3, annot_kws={"size": 7})
ax.set_title("Matrice di correlazione tra feature", fontsize=13)
plt.tight_layout()
savefig("05_correlation_matrix")


# ════════════════════════════════════════════════════════════════
# SEZIONE 5 — CLASSIFICAZIONE
# ════════════════════════════════════════════════════════════════
section("5. CLASSIFICAZIONE: Decision Tree, Naïve Bayes, SVM")

# Preparazione dati
feat_df = features[FEATURE_COLS + ["outcome"]].dropna()
X = feat_df[FEATURE_COLS]
y = feat_df["outcome"]

le = LabelEncoder()
y_enc = le.fit_transform(y)
class_names = le.classes_

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.25, stratify=y_enc, random_state=42)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print(f"Classi: {class_names}")

# ── Definizione modelli ──────────────────────────────────────────
models = {
    "Decision Tree": Pipeline([
        ("clf", DecisionTreeClassifier(max_depth=5, min_samples_leaf=5,
                                       class_weight="balanced", random_state=42))
    ]),
    "Naïve Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GaussianNB())
    ]),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.0, class_weight="balanced", random_state=42))
    ]),
}

# ── Cross-validation 5-fold ──────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results_cv = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y_enc, cv=cv, scoring="accuracy")
    results_cv[name] = scores
    print(f"{name:<20} CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

# ── Fit e valutazione sul test set ──────────────────────────────
print("\n--- Report classificazione sul test set ---")
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    fitted_models[name] = model
    acc = accuracy_score(y_test, y_pred)
    print(f"\n{name}  (Test accuracy: {acc:.4f})")
    print(classification_report(y_test, y_pred, target_names=class_names))

# ── Plot confusion matrices ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, fitted_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}\nTest acc: {accuracy_score(y_test, y_pred):.3f}")
plt.suptitle("Confusion Matrix — Test Set", fontsize=13, y=1.02)
plt.tight_layout()
savefig("06_confusion_matrices")

# ── Plot CV accuracy confronto ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
cv_df = pd.DataFrame(results_cv)
cv_df.boxplot(ax=ax)
ax.set_ylabel("Accuracy (5-fold CV)")
ax.set_title("Confronto modelli — Cross-Validation Accuracy")
ax.axhline(1/3, color="red", linestyle="--", alpha=0.5, label="Baseline casuale (3 classi)")
ax.legend()
plt.tight_layout()
savefig("07_cv_comparison")

# ── Feature importance (Decision Tree) ──────────────────────────
dt = fitted_models["Decision Tree"].named_steps["clf"]
importances = pd.Series(dt.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
importances.head(12).sort_values().plot.barh(ax=axes[0], color="#6366F1")
axes[0].set_title("Feature Importance — Decision Tree (top 12)")
axes[0].set_xlabel("Importance")

# Visualizzazione albero (depth=3 per leggibilità)
plot_tree(dt, feature_names=FEATURE_COLS, class_names=class_names,
          filled=True, max_depth=3, ax=axes[1], fontsize=7,
          impurity=False, proportion=True)
axes[1].set_title("Decision Tree (visualizzato fino a depth=3)")
plt.tight_layout()
savefig("08_decision_tree")


# ════════════════════════════════════════════════════════════════
# RIEPILOGO FINALE
# ════════════════════════════════════════════════════════════════
section("RIEPILOGO FINALE")

print(f"""
DATASET
  Fonte:       Premier League 2020-21 (Opta/WhoScored format)
  Righe:       607,656 eventi
  Partite:     380
  Squadre:     20 | Giocatori: ~524

FEATURE ESTRATTE ({len(FEATURE_COLS)})
  Conteggi:   n_passes, n_shots, n_shots_on_target, n_goals, n_tackles,
              n_fouls, n_corners, n_aerials, n_clearances,
              n_interceptions, n_takeons, n_cards, n_ball_recoveries
  Ratei:      pass_success_rate, shot_conversion_rate, shots_on_target_rate,
              tackle_success_rate, takeon_success_rate, aerial_success_rate
  Posizionali: avg_x, avg_y, avg_shot_x, avg_shot_y
  Intensità:  events_per_minute, first_half_ratio

FEATURE PIÙ DISCRIMINANTI (diff > 1σ tra Win e Loss):
""")
for feat in significant.index:
    print(f"  → {feat}")

print(f"""
CLASSIFICATORI
""")
for name, scores in results_cv.items():
    print(f"  {name:<20}  CV acc = {scores.mean():.4f} ± {scores.std():.4f}")

print(f"""
Plot salvati in: ./{OUT_DIR}/
""")


In [ ]:
df.head(50)

In [ ]:
# 1. Ispezione iniziale
print(f"Righe totali: {len(df)}")

# 2. Controllo Unicità Partite
# Nota: Non assegnare l'assert a una variabile.
assert df['match_id'].nunique() == 380, f"Errore: trovate {df['match_id'].nunique()} partite invece di 380"
print(f"✅ Numero di partite corretto: {df['match_id'].nunique()}")

# 3. Conferma 2 squadre per partita
teams_per_match = df.groupby('match_id')['teamId'].nunique()

# Questo assert controlla che OGNI partita abbia esattamente 2 squadre
assert (teams_per_match == 2).all(), \
    f'❌ Problema: {(teams_per_match != 2).sum()} partite non hanno 2 squadre!'

print("✅ Ogni partita ha esattamente 2 squadre.")
# Se vuoi vedere i primi risultati del raggruppamento:
print(teams_per_match.head()) 

# 4. Conferma numero totale squadre (Cardinalità)
n_squadre = df['teamId'].nunique()
assert n_squadre == 20, f"Errore: trovate {n_squadre} squadre invece di 20"
print(f"✅ Numero totale squadre corretto: {n_squadre}")

# Visualizza i primi 100 dati per scrupolo
df.head(100)

In [ ]:
print(df.columns.tolist())

In [ ]:
#event id 

# Cella di codice per "dimostrare" la scoperta al prof
# 1. Identifichiamo la partita
primo_id_partita = df['match_id'].unique()[0]
print(f"Analisi della partita ID: {primo_id_partita}")

# 2. Filtriamo
df_partita = df[df['match_id'] == primo_id_partita]

# 3. Verifichiamo la tua teoria dei range sulla partita 0
analisi_range = df_partita.groupby('teamId')['eventId'].agg(['min', 'max', 'count'])

print("\n--- ANALISI RANGE EVENT_ID PER SQUADRA ---")
print(analisi_range)

# 4. Anteprima cronologica usando 'Unnamed: 0' come contatore
# Ho aggiunto anche type_displayName così vedi che tipo di azione è (Pass, Tackle, etc.)
print("\n--- CRONOLOGIA EVENTI (Prime 15 righe) ---")
colonne_interessanti = ['Unnamed: 0', 'match_id', 'teamId', 'eventId', 'type_displayName', 'isTouch']
display(df_partita[colonne_interessanti].sort_values(by='Unnamed: 0').head(15))

In [ ]:
#qualifiers 
import pandas as pd
import ast

import pandas as pd
import ast

def estrai_tutti_i_qualifiers(dataframe):
    # Usiamo un 'set' per memorizzare i nomi in modo che non ci siano duplicati
    lista_completa = set()
    
    print("Analisi in corso... potrebbe volerci qualche secondo a seconda della dimensione del file.")

    for riga in dataframe['qualifiers']:
        # Se la riga è una stringa (comune nei CSV), convertila in oggetto Python
        if isinstance(riga, str):
            try:
                riga = ast.literal_eval(riga)
            except:
                continue
        
        # Estraiamo il displayName da ogni elemento della lista
        if isinstance(riga, list):
            for q in riga:
                nome = q.get('type', {}).get('displayName')
                if nome:
                    lista_completa.add(nome)
    
    # Convertiamo in lista ordinata alfabeticamente
    return sorted(list(lista_completa))

# Esecuzione
tutti_i_qualifiers = estrai_tutti_i_qualifiers(df)

print(f"\n--- TROVATI {len(tutti_i_qualifiers)} QUALIFIERS UNICI ---")
for q in tutti_i_qualifiers:
    print(f"- {q}")
    
# 1. Funzione per estrarre i nomi dei qualifiers in una lista leggibile
def get_qualifiers_list(q_list):
    # Se il dato è una stringa (succede a volte con i CSV), la convertiamo in lista
    if isinstance(q_list, str):
        try:
            q_list = ast.literal_eval(q_list)
        except:
            return []
    
    # Estraiamo solo il 'displayName' per ogni qualifier
    return [q.get('type', {}).get('displayName') for q in q_list if isinstance(q, dict)]

# 2. Creiamo una colonna temporanea semplificata per l'analisi
df['qualifiers_clean'] = df['qualifiers'].apply(get_qualifiers_list)

# 3. Vediamo quali sono i qualifiers più comuni nel dataset
from collections import Counter
all_qualifiers = [item for sublist in df['qualifiers_clean'] for item in sublist]
conteggio_qualifiers = Counter(all_qualifiers)

print("--- TOP 20 QUALIFIERS PIÙ FREQUENTI ---")
for qual, count in conteggio_qualifiers.most_common(20):
    print(f"{qual}: {count}")

# 4. Vediamo un esempio pratico per un tiro (isShot == True)
print("\n--- ESEMPIO DI QUALIFIERS PER UN TIRO ---")
esempio_tiro = df[df['isShot'] == True].iloc[0]
print(f"Evento: {esempio_tiro['type_displayName']}")
print(f"Qualifiers: {get_qualifiers_list(esempio_tiro['qualifiers'])}")

In [ ]:
# 1. Estraiamo solo le due colonne che ci interessano
# e usiamo drop_duplicates() per tenere solo una riga per ogni tipo di evento
legenda_eventi = df[['type_value', 'type_displayName']].drop_duplicates()

# 2. Ordiniamo la tabella in base al numero identificativo (type_value)
legenda_eventi = legenda_eventi.sort_values(by='type_value').reset_index(drop=True)

# 3. Mostriamo la legenda completa a schermo
print("--- LEGENDA UFFICIALE DEGLI EVENTI ---")
print(legenda_eventi.to_string(index=False))

# ---------------------------------------------------------
# BONUS "PRO": Creiamo un dizionario Python vero e proprio!
# Questo ti permetterà in futuro di tradurre un numero al volo.
# Es. dizionario_eventi[1] ti restituirà 'Pass'
# ---------------------------------------------------------
dizionario_eventi = dict(zip(legenda_eventi['type_value'], legenda_eventi['type_displayName']))

In [ ]:
# 1. Vediamo quante volte compaiono i nomi dei cartellini
occorrenze_nomi = df['cardType_displayName'].value_counts()
print("--- CONTEGGIO PER NOME CARTELLINO ---")
print(occorrenze_nomi)
print("\n") # Aggiunge uno spazio vuoto per leggere meglio

# 2. Creiamo la LEGENDA definitiva (Numero + Nome)
legenda_cartellini = df[['cardType_value', 'cardType_displayName']].dropna().drop_duplicates()

# Li ordiniamo dal numero più piccolo al più grande (31, 32, 33)
legenda_cartellini = legenda_cartellini.sort_values(by='cardType_value').reset_index(drop=True)

print("--- LEGENDA UFFICIALE CARTELLINI ---")
print(legenda_cartellini.to_string(index=False))